In [ ]:
# Giải thích bảng Vintage 31+ MOB
# Trục ngang là tenor (từ 0 đến kỳ hạn thanh toán cuối cùng), trục dọc là thời gian giải ngân tính theo tháng
# Các giá trị được tính theo các khoản vay có DPD > 30, thể hiện dưỡi dạng % với công thức: Tử = Tổng dư nợ DPD 31+ theo MOB, Mẫu = Tổng số tiền đã giải ngân trong tháng

In [2]:

import pandas as pd
import numpy as np

# 1. Đọc dữ liệu
file_path = r'D:\2. Đi học\Risk Analyst\File data\FACT_LOAN_MANAGEMENT.xlsx'

try:
    df_raw = pd.read_csv(file_path, sep='\t', encoding='latin-1')
except Exception:
    df_raw = pd.read_excel(file_path)

# 2. Xử lý tách cột
if len(df_raw.columns) == 1:
    col_name = df_raw.columns[0]
    df = df_raw[col_name].str.split('\t', expand=True)
    df.columns = col_name.split('\t')
else:
    df = df_raw.copy()

# 3. Chuẩn hóa tên cột và ép kiểu dữ liệu
df.columns = [str(col).strip().upper() for col in df.columns]
df['REPORT_DATE'] = pd.to_datetime(df['REPORT_DATE'], errors='coerce')
df['DISBURSEMENT_DATE'] = pd.to_datetime(df['DISBURSEMENT_DATE'], errors='coerce')
df['DPD'] = pd.to_numeric(df['DPD'], errors='coerce').fillna(0)
df['PRINCIPAL_OUTSTANDING'] = pd.to_numeric(df['PRINCIPAL_OUTSTANDING'], errors='coerce').fillna(0)
df['LOAN_AMOUNT'] = pd.to_numeric(df['LOAN_AMOUNT'], errors='coerce').fillna(0)
df['TENOR'] = pd.to_numeric(df['TENOR'], errors='coerce').fillna(0)

df = df.dropna(subset=['REPORT_DATE', 'DISBURSEMENT_DATE'])

# 4. Định dạng và tính MOB
df['DISBURSEMENT_MONTH'] = df['DISBURSEMENT_DATE'].dt.strftime('%Y-%m')
df['MOB'] = ((df['REPORT_DATE'].dt.year - df['DISBURSEMENT_DATE'].dt.year) * 12 + 
             (df['REPORT_DATE'].dt.month - df['DISBURSEMENT_DATE'].dt.month))

# --- TÍNH TOÁN TỶ LỆ VINTAGE % ---

# Bước A: Tính tổng giải ngân ban đầu theo từng tháng (Mẫu số)
total_disbursement = df.groupby('DISBURSEMENT_MONTH')['LOAN_AMOUNT'].sum()

# Bước B: Tính tổng dư nợ DPD 31+ theo tháng và MOB (Tử số)
df_31plus = df[df['DPD'] > 30].copy()
bad_debt_pivot = pd.pivot_table(
    df_31plus, 
    values='PRINCIPAL_OUTSTANDING', 
    index='DISBURSEMENT_MONTH', 
    columns='MOB', 
    aggfunc='sum'
).fillna(0)

# Bước C: Chia Tử số cho Mẫu số để ra tỷ lệ %
vintage_percentage = bad_debt_pivot.div(total_disbursement, axis=0)

# 5. Ép Header từ 0 đến Tenor cuối cùng
max_tenor = int(df['TENOR'].max())
all_mobs = list(range(0, max_tenor + 1))
vintage_final = vintage_percentage.reindex(columns=all_mobs, fill_value=0)

# 6. Xuất ra Excel
output_path = r'D:\2. Đi học\Risk Analyst\File data\VINTAGE_PERCENTAGE_31PLUS.xlsx'
vintage_final.to_excel(output_path)

print("--- ĐÃ HOÀN THÀNH BẢNG VINTAGE TỶ LỆ % ---")
print(f"File lưu tại: {output_path}")
print(vintage_final.head())

--- ĐÃ HOÀN THÀNH BẢNG VINTAGE TỶ LỆ % ---
File lưu tại: D:\2. Đi học\Risk Analyst\File data\VINTAGE_PERCENTAGE_31PLUS.xlsx
MOB                 0    1         2         3         4         5         6   \
DISBURSEMENT_MONTH                                                              
2022-01              0  0.0  0.010447  0.026110  0.027086  0.026212  0.027086   
2022-02              0  0.0  0.008888  0.024747  0.023948  0.024747  0.024747   
2022-03              0  0.0  0.016260  0.026754  0.027646  0.027646  0.026754   
2022-04              0  0.0  0.013035  0.031639  0.031639  0.030619  0.031639   
2022-05              0  0.0  0.013906  0.027952  0.027050  0.027952  0.027050   

MOB                       7         8         9   ...  27  28  29  30  31  32  \
DISBURSEMENT_MONTH                                ...                           
2022-01             0.027086  0.026212  0.027086  ...   0   0   0   0   0   0   
2022-02             0.023948  0.024747  0.023948  ...   0   0   0

In [3]:
display(vintage_final)

MOB,0,1,2,3,4,5,6,7,8,9,...,27,28,29,30,31,32,33,34,35,36
DISBURSEMENT_MONTH,,,,,,,,,,,,,,,,,,,,,
2022-01,0,0.000000,0.010447,0.026110,0.027086,0.026212,0.027086,0.027086,0.026212,0.027086,...,0,0,0,0,0,0,0,0,0,0
2022-02,0,0.000000,0.008888,0.024747,0.023948,0.024747,0.024747,0.023948,0.024747,0.023948,...,0,0,0,0,0,0,0,0,0,0
2022-03,0,0.000000,0.016260,0.026754,0.027646,0.027646,0.026754,0.027646,0.026754,0.027646,...,0,0,0,0,0,0,0,0,0,0
2022-04,0,0.000000,0.013035,0.031639,0.031639,0.030619,0.031639,0.030619,0.031639,0.031639,...,0,0,0,0,0,0,0,0,0,0
2022-05,0,0.000000,0.013906,0.027952,0.027050,0.027952,0.027050,0.027952,0.027952,0.025247,...,0,0,0,0,0,0,0,0,0,0
2022-06,0,0.000000,0.022076,0.037510,0.038760,0.037510,0.038760,0.038760,0.035009,0.038760,...,0,0,0,0,0,0,0,0,0,0
2022-07,0,0.000144,0.020609,0.037280,0.036078,0.037280,0.037280,0.033672,0.037280,0.036078,...,0,0,0,0,0,0,0,0,0,0
2022-08,0,0.000000,0.018683,0.033460,0.034576,0.034576,0.031230,0.034576,0.033460,0.034576,...,0,0,0,0,0,0,0,0,0,0
2022-09,0,0.000000,0.021581,0.038884,0.038884,0.035121,0.038884,0.037630,0.038884,0.037630,...,0,0,0,0,0,0,0,0,0,0
